# Lab 5: Requests and JSON with Python

## Mac Jackson - SI 676 - Fall 2026

HTTP Requests with Python

### 1. URL / URI Formulation
Given the following lists of resources, form the URIs that will request the pages for each of the following items:

1. /resource/cph.3f05183/
2. /resource/fsa.8d24709/
3. /resource/highsm.64003/

Hint: As noted in class, you can locate these items by appending the resource identifiers to https://www.loc.gov.

Answer:
1. https://www.loc.gov./resource/cph.3f05183/
2. https://www.loc.gov./resource/fsa.8d24709/
3. https://www.loc.gov./resource/highsm.64003/

In [33]:
import requests

base_url = "https://www.loc.gov."  # Base URL for the Library of Congress API
resource_urls = [
    "/resource/cph.3f05183/",
    "/resource/fsa.8d24709/",
    "/resource/highsm.64003/"
]

uris = [base_url + url for url in resource_urls]

for uri in uris:
    response = requests.get(uri)
    print(uri)

https://www.loc.gov./resource/cph.3f05183/
https://www.loc.gov./resource/fsa.8d24709/
https://www.loc.gov./resource/highsm.64003/


### 2. LC Permalinks as URIs

The Library of Congress provides a standard “permalink” structure, which is intended to be a stable identifier that can provide a consistent URI for any item in the Library’s collections. The pattern for these URIs is https://lccn.loc.gov/ + LCCN.

Using the URIs that you created in the previous example, identify an LC permalink for each of the resources. To do this, look at the information provided about each resource, identify the LCCN, and then create the permalink.

In [34]:
# find permalink for each resource
for uri in uris:
    r = requests.get(uri, params={"fo": "json"})
    data = r.json()

    lccn = data["item"]["number_lccn"]   # adjust if the key differs
    print(uri, "+", lccn)

    lccn = str(lccn)  # Convert to string bc cant concatenate

    print("Permalink:", "https://lccn.loc.gov/" + lccn[2:-2])  # Assuming the LCCN is at least 8 characters long


https://www.loc.gov./resource/cph.3f05183/ + ['98508155']
Permalink: https://lccn.loc.gov/98508155
https://www.loc.gov./resource/fsa.8d24709/ + ['2017843202']
Permalink: https://lccn.loc.gov/2017843202
https://www.loc.gov./resource/highsm.64003/ + ['2020722343']
Permalink: https://lccn.loc.gov/2020722343


### 3. Retrieve JSON data for items

The loc.gov website allows for responses to be requested in a data format, like JSON. Using the same three items as above, write a block of Python code using the requests library that will request and retrieve the JSON data about the items above, then save them in local JSON files.

Recall: you can provide the parameter fo with the value of json to request the page “data” in JSON format.

In [35]:
import requests
import json
import time
from pathlib import Path

#same as # 1
base_url = "https://www.loc.gov."  # Base URL for the Library of Congress API
resource_urls = [
    "/resource/cph.3f05183/",
    "/resource/fsa.8d24709/",
    "/resource/highsm.64003/"
]

for path in resource_urls:
    r = requests.get(base_url + path, params={"fo": "json"})

    print('You requested:',r.url)
    print('HTTP server response code:',r.status_code)
    print('HTTP response headers',r.headers)
    print('\nYour request has this content type:\n',r.headers['content-type'])

    r.raise_for_status()
    data = r.json()

    filename = Path(path).name + ".json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)

    print(f"Data saved to {filename}")
    time.sleep(1)  # Avoid overwhelming the server aka getting blocked




#r.text


You requested: https://www.loc.gov./resource/cph.3f05183/?fo=json
HTTP server response code: 200
HTTP response headers {'Date': 'Fri, 30 Oct 2026 01:44:51 GMT', 'Content-Type': 'application/json', 'Content-Length': '44814', 'Connection': 'keep-alive', 'access-control-allow-origin': '*', 'referrer-policy': 'no-referrer-when-downgrade', 'strict-transport-security': 'max-age=3600; preload', 'x-content-type-options': 'nosniff', 'x-robots-tag': 'noindex, nofollow', 'x-frame-options': 'sameorigin', 'Age': '5775', 'etag': '"3f2eeb30c67d81b8d4ed860db8d3f479"', 'expires': 'Sat, 31 Oct 2026 00:08:35 GMT', 'content-security-policy': "block-all-mixed-content;         default-src https://loc.gov/ https://*.loc.gov/ ;         media-src https://loc.gov/ https://*.loc.gov/              https://*.readspeaker.com/             https://*.arcgis.com/ https://*.arcgisonline.com/  https://webapps-cdn.esri.com/             blob:;         worker-src https://loc.gov/ https://*.loc.gov/              blob:;      

### 4. Use a different parameter

Assume that the https://www.loc.gov/search address at the LOC can receive search requests by appending the parameter q to the URL. This is a way to send a search query automatically via the URL (think of the variable q for “query”). Create a short block of Python code that queries the base URL. Use the input() function so the code will ask the user for a unique query, save that query as a string that will be inserted into the URI request as a parameter, send the request using python requests, and return the results. The script should print out the results to the terminal or in the Jupyter notebook. (In a production environment you would likely want to display such results in a browser or perhaps save them in a JSON file.)

In [36]:
import requests

searchEndpoint = 'http://www.loc.gov/search/'

parameters = {
    'fo' : 'json',
    'q'  : input("Enter your search query: "),
    'c' : 10 # limit the number of results returned
}

r = requests.get(searchEndpoint, params = parameters)

print('You requested:',r.url)
print('HTTP server response code:',r.status_code)
print('HTTP response headers',r.headers)
print('\nYour request has this content type:\n',r.headers['content-type'])

r.raise_for_status()
data = r.json()

# print results
for item in data['results']:
    print(f"Title: {item['title']}")
    print(f"URL: {item['url']}")
    print(f"Date: {item.get('date', 'N/A')}")
    print(f"Description: {item.get('description', 'No description available.')}")
    print("-" * 40)

You requested: https://www.loc.gov/search/?c=10&fo=json&q=rupaul%27s+drag+race
HTTP server response code: 200
HTTP response headers {'Date': 'Fri, 30 Oct 2026 01:45:07 GMT', 'Content-Type': 'application/json', 'Content-Length': '62636', 'Connection': 'keep-alive', 'access-control-allow-origin': '*', 'referrer-policy': 'no-referrer-when-downgrade', 'strict-transport-security': 'max-age=3600; preload', 'x-content-type-options': 'nosniff', 'x-robots-tag': 'noindex, nofollow', 'x-frame-options': 'sameorigin', 'etag': '"4c29d8f5200b2f5a66f64c0a2f9770fc"', 'expires': 'Sat, 31 Oct 2026 01:45:03 GMT', 'content-security-policy': "block-all-mixed-content;         default-src https://loc.gov/ https://*.loc.gov/ ;         media-src https://loc.gov/ https://*.loc.gov/              https://*.readspeaker.com/             https://*.arcgis.com/ https://*.arcgisonline.com/  https://webapps-cdn.esri.com/             blob:;         worker-src https://loc.gov/ https://*.loc.gov/              blob:;        